# Synthea FHIR Output Analysis

Analyse the patient bundles generated by Synthea to inform cleanup decisions.

In [21]:
from pathlib import Path

# Folder containing the Synthea FHIR patient bundles to analyse
PATIENTS_DIR: Path = Path("~/Developer/misc/synthea/output/fhir/patients")
DECEASED_PATIENTS_DIR: Path = Path("~/Developer/misc/synthea/output/fhir/deceased_patients")
print(f"PATIENTS_DIR: {PATIENTS_DIR.expanduser()}")
print(f"DECEASED_PATIENTS_DIR: {DECEASED_PATIENTS_DIR.expanduser()}")

PATIENTS_DIR: /Users/8826/Developer/misc/synthea/output/fhir/patients
DECEASED_PATIENTS_DIR: /Users/8826/Developer/misc/synthea/output/fhir/deceased_patients


In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import HBox, VBox, interactive_output
from tqdm.notebook import tqdm

files: list[Path] = sorted(PATIENTS_DIR.expanduser().glob("*.json"), key=lambda f: f.name)
sizes_mb: list[float] = [f.stat().st_size / (1024 * 1024) for f in files]

print(f"Total files : {len(files)}")
print(f"Min size    : {min(sizes_mb):.2f} MB" if sizes_mb else "No files found")
print(f"Max size    : {max(sizes_mb):.2f} MB")
print(f"Median size : {np.median(sizes_mb):.2f} MB")
print(f"Mean size   : {np.mean(sizes_mb):.2f} MB")

bins_slider = widgets.IntSlider(value=50, min=5, max=200, step=5, description="Bins")
threshold_slider = widgets.FloatSlider(
    value=round(np.median(sizes_mb), 2),
    min=round(min(sizes_mb), 2),
    max=round(max(sizes_mb), 2),
    step=0.01,
    description="Threshold (MB)",
    style={"description_width": "initial"},
    readout_format=".2f",
)

def plot_histogram(bins: int, threshold: float) -> None:
    below = sum(1 for s in sizes_mb if s <= threshold)
    above = len(sizes_mb) - below

    _, ax = plt.subplots(figsize=(30, 4))
    ax.hist(sizes_mb, bins=bins, edgecolor="white", linewidth=0.4)
    ax.axvline(threshold, color="crimson", linewidth=1.5, linestyle="--", label=f"Threshold: {threshold:.2f} MB")
    ax.set_xlabel("File size (MB)")
    ax.set_ylabel("Number of files")
    ax.set_title(
        f"Synthea patient bundle size distribution  |  "
        f"≤ {threshold:.2f} MB: {below} files   >  {threshold:.2f} MB: {above} files"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()

out = interactive_output(plot_histogram, {"bins": bins_slider, "threshold": threshold_slider})
display(VBox([HBox([bins_slider, threshold_slider]), out]))


Total files : 1000
Min size    : 0.46 MB
Max size    : 23.06 MB
Median size : 1.28 MB
Mean size   : 2.43 MB


In [26]:
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from ipywidgets import HBox, VBox, interactive_output

cumulative_mb: list[float] = list(np.cumsum(sizes_mb))

index_slider = widgets.IntSlider(
    value=len(files),
    min=1,
    max=len(files),
    step=1,
    description="Index (n)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

size_threshold_slider = widgets.FloatSlider(
    value=round(np.median(sizes_mb), 2),
    min=round(min(sizes_mb), 2),
    max=round(max(sizes_mb), 2),
    step=0.01,
    description="Max file size (MB)",
    style={"description_width": "initial"},
    readout_format=".2f",
    layout=widgets.Layout(width="600px"),
)

index_label = widgets.HTML()
filter_label = widgets.HTML()

def update_index(n: int, size_threshold: float) -> None:
    total = cumulative_mb[n - 1]
    index_label.value = (
        f"<b>First {n} files</b> (alphabetical) — "
        f"cumulated size: <b>{total:.2f} MB</b> &nbsp;|&nbsp; "
        f"remaining {len(files) - n} files: <b>{cumulative_mb[-1] - total:.2f} MB</b>"
    )
    small = [(f, s) for f, s in zip(files, sizes_mb) if s <= size_threshold]
    small_total = sum(s for _, s in small)
    filter_label.value = (
        f"Files ≤ <b>{size_threshold:.2f} MB</b>: <b>{len(small)}</b> files &nbsp;|&nbsp; "
        f"cumulated size: <b>{small_total:.2f} MB</b>"
    )

out = interactive_output(update_index, {"n": index_slider, "size_threshold": size_threshold_slider})
display(VBox([HBox([index_slider, size_threshold_slider]), index_label, filter_label, out]))

In [ ]:
%%bash
mkdir -p /Users/8826/Developer/misc/synthea/output/fhir/too_heavy

find /Users/8826/Developer/misc/synthea/output/fhir/patients \
  -maxdepth 1 -name "*.json" -size +12M \
  -exec mv {} /Users/8826/Developer/misc/synthea/output/fhir/too_heavy/ \;

echo "Moved $(ls /Users/8826/Developer/misc/synthea/output/fhir/too_heavy | wc -l | tr -d ' ') files to too_heavy/"
echo "Remaining in patients/: $(ls /Users/8826/Developer/misc/synthea/output/fhir/patients | wc -l | tr -d ' ') files"

## Optional: Move deceased patients to a separate folder

In [24]:
import shutil

dest = DECEASED_PATIENTS_DIR.expanduser()
dest.mkdir(parents=True, exist_ok=True)

alive, dead = 0, 0
for f in tqdm(files, desc="Reading patients"):
    bundle = json.loads(f.read_text())
    patient = next(
        e["resource"] for e in bundle["entry"]
        if e["resource"].get("resourceType") == "Patient"
    )
    if patient.get("deceasedBoolean") or patient.get("deceasedDateTime"):
        shutil.move(str(f), dest / f.name)
        dead += 1
    else:
        alive += 1
print(f"\nAlive       : {alive}")
print(f"Dead        : {dead} (moved to {dest})")


Reading patients:   0%|          | 0/1368 [00:00<?, ?it/s]


Alive       : 1000
Dead        : 368 (moved to /Users/8826/Developer/misc/synthea/output/fhir/deceased_patients)
